In [1]:
import sys
sys.path.append("../")

from pathlib import Path

from src.evaluating import Evaluator
from src.llm.loading import Loader
from src.ner.masking import apply_masking
from src.ner.regex_detector import RegexDetector
from src.reporting import write_baseline_report


/Users/ayeustsihneyeu/PythonProjects/mask_sft/.masksft/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
loader = Loader(path=Path("../data/data.json"), test_k=0.2, seed=42)
tests = loader.load_test_dataset()
tests


Dataset({
    features: ['input', 'target'],
    num_rows: 10
})

In [3]:
detector = RegexDetector()
model_name = "Regex NER Baseline"
model_name


'Regex NER Baseline'

In [4]:
result = []
for test in tests:
    predicted_entities = detector.detect(test["input"])
    masked_prediction = apply_masking(test["input"], predicted_entities)
    result.append({
        "input": test["input"],
        "prediction": masked_prediction,
        "target": test["target"],
        "predicted_entities": predicted_entities,
    })

len(result)


10

In [5]:
evaluator = Evaluator(
    predictions=[row["prediction"] for row in result],
    references=[row["target"] for row in result],
)

metrics = evaluator.evaluate()
metrics


{'samples': 10,
 'exact_match': 0.2,
 'masking_recall': 0.5,
 'over_masking_rate': 0.10526315789473684,
 'text_preservation': 0.9184594910085944}

In [6]:
report_path = write_baseline_report(
    model_name=model_name,
    metrics=metrics,
    predictions=result,
    output_path="../reports/08_ner_regex_baseline.md",
    preview_size=len(result),
)

print(f"Regex NER baseline report saved to {report_path}")

preview = result[:3]
for index, row in enumerate(preview, start=1):
    print()
    print(f"Sample {index}")
    print("INPUT:", row["input"])
    print("PREDICTION:", row["prediction"])
    print("TARGET:", row["target"])
    print("ENTITIES:", row["predicted_entities"])


Regex NER baseline report saved to ../reports/08_ner_regex_baseline.md

Sample 1
INPUT: Hello Mrs. Walker, can you check why my recent transfer of $250 hasn't been processed? My account number is 567890 and my email is megan.walker@example.com.
PREDICTION: Hello Mrs. Walker, can you check why my recent transfer of [CURRENCYSYMBOL]250 hasn't been processed? My account number is [ACCOUNT_NUMBER] and my email is [EMAIL].
TARGET: Hello [PREFIX], can you check why my recent transfer of [AMOUNT] hasn't been processed? My account number is [ACCOUNT_NUMBER] and my email is [EMAIL].
ENTITIES: [{'start': 59, 'end': 60, 'label': 'CURRENCYSYMBOL'}, {'start': 108, 'end': 114, 'label': 'ACCOUNT_NUMBER'}, {'start': 131, 'end': 155, 'label': 'EMAIL'}]

Sample 2
INPUT: Dear Ms. Garcia, I need to report a lost debit card. My phone number is (555) 789-0123 and my account number is 678901.
PREDICTION: Dear Ms. Garcia, I need to report a lost debit card. My phone number is (555) [PHONE_NUMBER] and my accou